## Sensitivity Analysis of W&B Runs

In [ ]:
import wandb

# Connect to Wandb
wandb.login()

# Specify the project name
project_name = "karman"

# Specify the date after which you want to retrieve the run IDs
start_date = "2024-08-04T19:12:00"      # August 4th, 2024 at 8:12 pm

# Get the runs for the specified project and date
api = wandb.Api()

runs = api.runs(path="karman", filters={"createdAt": {"$gte": start_date}}) #, created_after=start_date)

# Extract the run IDs
run_ids = [run.id for run in runs]

# Print the run IDs
print(run_ids)

# Check these are all past start_date, will error if not
assert(all([r.createdAt > start_date for r in runs]))

In [ ]:
run = runs[0]
valid_keys = [k for k in run.summary.keys() if "valid_total" in k]
valid_history = run.history(keys=["_step"]+valid_keys) 

train_keys = [k for k in run.summary.keys() if "train_total" in k]
train_history = run.history(keys=["_step"]+train_keys) 

In [ ]:
for c in train_history.columns:
    if c == "_step":
        continue
    print(c, train_history[c].min(), train_history[c].argmin())
    # print(train_history[c][], train_history[c].argmax())

In [ ]:
import pandas as pd

run = runs[0]
valid_keys = [k for k in run.summary.keys() if "valid_total" in k]
train_keys = [k for k in run.summary.keys() if "train_total" in k]

# Create an empty DataFrame to store the results
results_df = pd.DataFrame(columns=['Run ID'] + train_keys + valid_keys)

results = []

# Iterate over each run
for run in runs:
    # Get the train_history and valid_history for the run
    train_history = run.history(keys=["_step"]+train_keys)
    valid_history = run.history(keys=["_step"]+valid_keys)

    # Should really do this with argmin on one and take the values from the rest!!
    train_arg_min = train_history["nn_mape_train_total"].argmin()
    valid_arg_min = valid_history["nn_mape_valid_total"].argmin()

    result = {'Run ID': run.id, "Run Name": run.name.replace("run_gpu", "").replace("_epochs_"+str(run.config["epochs"]), ""), "Lag": None, "Epochs": run.config["epochs"],
                                    **{key: train_history[key][train_arg_min] for key in train_keys if key not in ["_step", "q_loss_train_total", "q_loss_valid_total"]},
                                    **{key: valid_history[key][valid_arg_min] for key in valid_keys if key not in ["_step", "q_loss_train_total", "q_loss_valid_total"]}
    }
    if "lag_minutes" in run.config:
        result["Lag"] = run.config["lag_minutes"]

    results.append(result)

# Print the results
# print(results_df)
pd.DataFrame(results).sort_values("nn_mape_valid_total")

In [ ]:
pd.DataFrame(results).sort_values("nn_mape_valid_total").to_csv("results.csv")

In [ ]:
run.config

In [ ]:
runs[0].config['lag_minutes']

In [ ]:
import pandas as pd

# Create an empty DataFrame to store the results
results_df = pd.DataFrame(columns=['Run ID'] + train_keys + valid_keys)

# Iterate over each run
for run in runs:
    # Get the train_history and valid_history for the run
    train_history = run.history(keys=["_step"]+train_keys)
    valid_history = run.history(keys=["_step"]+valid_keys)

    results_df = results_df.append({'Run ID': run.id, 
                                    **{key: train_history[key].min() for key in train_keys if key not "_step"},
                                    **{key: valid_history[key].min() for key in valid_keys if ket not "_step"}
    }, ignore_index=True)

# Print the results
print(results_df)

In [ ]:
history = run.scan_history(keys=["_step", "nn_mape_valid"])
# for i in history:
#     if i["_step"] == 100:
#         print(i["nn_mape_train"])#, i["nn_mape_valid"])
#         break
history.shape

In [ ]:
history.shape

In [ ]:
[k for k in runs[0].summary.keys() if "_total" in k]

In [ ]:
run.summary["nn_mape_valid_total"]
